# Practice Lab 5.2 - LangGraph for stateful, multi-step workflows (WORKED solutions)

**Course:** Netsetos FDE Cohort 1 - Module 5 - Lesson 5.2 - Practice Lab

This notebook is the **worked-solution companion** to `practice-lab-5.2.html` (8 exercises, 130 points).
Each exercise below repeats the task and success criteria, then shows a runnable worked solution. Run it top to bottom.

- **Keyless-runnable.** The setup cell installs a deterministic offline fallback engine, so every graph still
  builds, routes, checkpoints, interrupts, and resumes with **no `ANTHROPIC_API_KEY` and no network**. Set a key
  (and `pip install` the real langgraph stack) to run the classify / propose nodes on Sonnet 5.
- Exercises 1-3 are pure Python (no model). Exercises 4-8 build AcmeCorp's escalation graph; a few use
  real-langgraph features (`SqliteSaver` on disk, `update_state`, the `Store`, token streaming) that degrade to a
  printed note under the offline engine and reproduce fully with `pip install "langgraph>=1.0"`.
- Production context: AcmeCorp customer-support escalation, USD refund amounts, $500 human-approval threshold.


In [ ]:
!pip install --quiet "langgraph>=1.0" "langgraph-checkpoint-sqlite>=2.0" "langchain-anthropic>=0.3" langchain-core anthropic

In [ ]:
# Reproducibility - the lesson uses seeded random values throughout.
# This notebook is KEYLESS-RUNNABLE: it executes end-to-end with NO
# ANTHROPIC_API_KEY / OPENAI_API_KEY set. When a real key is present the LLM
# nodes call the live model; otherwise a deterministic canned model stands in
# so every graph still routes, checkpoints, and resumes exactly the same way.
import os
import re
import random
import hashlib
import tempfile

random.seed(42)

HAS_ANTHROPIC = bool(os.environ.get("ANTHROPIC_API_KEY"))
HAS_OPENAI = bool(os.environ.get("OPENAI_API_KEY"))
print(f"HAS_ANTHROPIC={HAS_ANTHROPIC}  HAS_OPENAI={HAS_OPENAI}")

# --- Framework imports (guarded) --------------------------------------------
# If LangGraph / LangChain are unavailable or need network at import time, we
# fall back to a compact in-notebook engine so the graph still runs offline.
try:
    from typing import Annotated, Literal, TypedDict
    from langchain_core.messages import AIMessage, BaseMessage, HumanMessage
    from langgraph.graph import END, START, StateGraph
    from langgraph.graph.message import add_messages
    from langgraph.checkpoint.memory import InMemorySaver
    from langgraph.types import Command, interrupt
    LANGGRAPH_AVAILABLE = True
except Exception as exc:  # pragma: no cover - offline fallback engine
    LANGGRAPH_AVAILABLE = False
    print(f"[offline] LangGraph unavailable ({exc!r}); using in-notebook fallback")
    from typing import Annotated, Literal, TypedDict  # stdlib, always present

    class BaseMessage:
        def __init__(self, content="", **kw):
            self.content = content
            self.__dict__.update(kw)
        def __repr__(self):
            return f"{type(self).__name__}({self.content!r})"

    class HumanMessage(BaseMessage):
        pass

    class AIMessage(BaseMessage):
        pass

    def add_messages(left, right):
        left = list(left or [])
        right = right if isinstance(right, list) else [right]
        return left + right

    START, END = "__start__", "__end__"

    class Command:
        def __init__(self, resume=None):
            self.resume = resume

    class _Interrupt(Exception):
        def __init__(self, value):
            super().__init__("interrupt")
            self.value = value

    _RESUME = {}
    _CURRENT = {"tid": None}

    def interrupt(payload):
        tid = _CURRENT["tid"]
        if tid in _RESUME:
            return _RESUME.pop(tid)
        raise _Interrupt(payload)

    class InMemorySaver:
        def __init__(self, *a, **k):
            self.store = {}

    class _Snapshot:
        def __init__(self, config, values):
            self.config = config
            self.values = values

    class _App:
        def __init__(self, nodes, edges, cond, reducers, checkpointer):
            self.nodes, self.edges, self.cond = nodes, edges, cond
            self.reducers = reducers
            self.cp = checkpointer or InMemorySaver()
        def _apply(self, state, update):
            for k, v in (update or {}).items():
                state[k] = self.reducers[k](state.get(k), v) if k in self.reducers else v
            return state
        def _next(self, node, state):
            if node in self.cond:
                router, mapping = self.cond[node]
                return mapping[router(state)]
            return self.edges.get(node, END)
        def _persist(self, tid, state):
            lst = self.cp.store.setdefault(tid, [])
            lst.append(_Snapshot({"configurable": {"thread_id": tid, "checkpoint_id": len(lst)}}, dict(state)))
        def invoke(self, inp, config=None):
            tid = (config or {}).get("configurable", {}).get("thread_id", "default")
            _CURRENT["tid"] = tid
            if isinstance(inp, Command):
                _RESUME[tid] = inp.resume
                state = dict(self.cp.store[tid][-1].values)
                node = state.pop("__pending__", END)
            else:
                state = dict(inp or {})
                self._persist(tid, state)
                node = self.edges.get(START, END)
            while node != END:
                try:
                    update = self.nodes[node](state)
                except _Interrupt as it:
                    state["__pending__"] = node
                    self._persist(tid, state)
                    return {**{k: v for k, v in state.items() if k != "__pending__"}, "__interrupt__": (it.value,)}
                state = self._apply(state, update)
                self._persist(tid, state)
                node = self._next(node, state)
            state.pop("__pending__", None)
            return dict(state)
        def stream(self, inp, config=None, stream_mode="updates"):
            tid = (config or {}).get("configurable", {}).get("thread_id", "default")
            _CURRENT["tid"] = tid
            state = dict(inp or {})
            self._persist(tid, state)
            node = self.edges.get(START, END)
            while node != END:
                update = self.nodes[node](state)
                state = self._apply(state, update)
                self._persist(tid, state)
                yield {node: update}
                node = self._next(node, state)
        def get_state_history(self, config=None):
            tid = (config or {}).get("configurable", {}).get("thread_id", "default")
            return list(reversed(self.cp.store.get(tid, [])))

    class StateGraph:
        def __init__(self, state_type):
            self.nodes, self.edges, self.cond, self.reducers = {}, {}, {}, {}
            for k, ann in getattr(state_type, "__annotations__", {}).items():
                meta = getattr(ann, "__metadata__", None)
                if meta:
                    self.reducers[k] = meta[0]
        def add_node(self, name, fn):
            self.nodes[name] = fn
        def add_edge(self, a, b):
            self.edges[a] = b
        def add_conditional_edges(self, src, router, mapping):
            self.cond[src] = (router, mapping)
        def compile(self, checkpointer=None):
            return _App(self.nodes, self.edges, self.cond, self.reducers, checkpointer)


def make_model(canned):
    # Return a chat model. Live ChatAnthropic when ANTHROPIC_API_KEY is set;
    # otherwise a deterministic FakeListChatModel (or a tiny local shim) that
    # returns the canned responses - so LLM nodes produce sensible AIMessages
    # with no key and no network.
    if HAS_ANTHROPIC:
        from langchain_anthropic import ChatAnthropic
        return ChatAnthropic(model="claude-sonnet-5", timeout=30.0)
    if LANGGRAPH_AVAILABLE:
        # Real LangChain stack present -> use its deterministic fake chat model.
        try:
            from langchain_core.language_models.fake_chat_models import FakeListChatModel
            return FakeListChatModel(responses=list(canned))
        except Exception:
            pass
    # Offline / fallback engine active -> tiny shim over our AIMessage class.
    class _FakeModel:
        def __init__(self, responses):
            self._responses = list(responses) or [""]
            self._i = 0
        def invoke(self, _messages):
            text = self._responses[self._i % len(self._responses)]
            self._i += 1
            return AIMessage(content=text)
    return _FakeModel(canned)

---
## Exercise 1 - The minimum graph that is still a graph

**Task.** Build a StateGraph whose state has a single messages field wired with the add_messages reducer, one chatbot node, and edges START -> chatbot -> END. Invoke it twice on the same result to prove the reducer APPENDS instead of overwriting.

**Success criteria**
- After two invocations the messages list holds BOTH turns (4+ messages), not just the last AI reply
- Swapping the annotation for a plain list[BaseMessage] collapses the count to 1 - the number-one LangGraph footgun (Hamel Husain field guide, failure mode #1) - and you can explain why


In [ ]:
# Worked solution - the minimum graph that is still a graph.
# (BaseMessage/HumanMessage/AIMessage/add_messages/START/END/StateGraph and make_model
#  all come from the keyless setup cell above.)
from typing import Annotated, TypedDict

class ChatState(TypedDict):
    # add_messages is the reducer that APPENDS instead of overwriting the list.
    messages: Annotated[list, add_messages]

model = make_model(canned=["Hi from AcmeCorp support - how can I help?"])

def chatbot(state: ChatState) -> dict:
    reply = model.invoke(state["messages"])
    return {"messages": [reply]}          # partial update; add_messages appends

graph = StateGraph(ChatState)
graph.add_node("chatbot", chatbot)
graph.add_edge(START, "chatbot")
graph.add_edge("chatbot", END)
app = graph.compile()

# First turn:
r1 = app.invoke({"messages": [HumanMessage("hi")]})
# Feed the accumulated messages back in and invoke again:
r2 = app.invoke({"messages": r1["messages"] + [HumanMessage("one more thing")]})

print("after run 1:", len(r1["messages"]), "messages")
print("after run 2:", len(r2["messages"]), "messages")
assert len(r2["messages"]) >= 4, "add_messages must APPEND both turns, not overwrite"
print("OK - reducer appended; a plain list[BaseMessage] annotation would collapse this to 1")

---
## Exercise 2 - Five fields, five reducer decisions

**Task.** Write the EscalationState TypedDict with the six fields from the lesson (messages, customer_id, customer_tier, intent, refund_amount_usd, approval_status). Give messages the add_messages reducer; the scalars keep the default overwrite reducer (correct for them). Then write a 5-line unit test that runs a two-node graph and asserts the message list grew.

**Success criteria**
- Six fields declared; messages is Annotated[list[BaseMessage], add_messages], the scalars keep the default overwrite reducer
- The append test passes only when messages carries the add_messages reducer
- Full-marks answers add a one-line comment on EACH field naming its reducer mode - explicit per-field decisions kill roughly 60% of production LangGraph bugs


In [ ]:
# Worked solution - EscalationState with an explicit reducer decision per field.
from typing import Annotated, Literal, TypedDict

class EscalationState(TypedDict):
    messages: Annotated[list, add_messages]                              # APPEND (+ dedup by id)
    customer_id: str                                                     # overwrite (default)
    customer_tier: Literal["free", "pro", "enterprise", "unknown"]       # overwrite
    intent: Literal["refund_request", "billing_question", "feature_help", "unknown"]  # overwrite
    refund_amount_usd: float | None                                      # overwrite
    approval_status: Literal["pending", "approved", "rejected", None]    # overwrite

# 5-line unit test: a two-node graph, then assert the message list grew.
def node_a(state: EscalationState) -> dict:
    return {"messages": [AIMessage("ack from A")], "intent": "refund_request"}

def node_b(state: EscalationState) -> dict:
    return {"messages": [AIMessage("ack from B")]}   # scalars untouched -> not clobbered

g = StateGraph(EscalationState)
g.add_node("a", node_a); g.add_node("b", node_b)
g.add_edge(START, "a"); g.add_edge("a", "b"); g.add_edge("b", END)
app = g.compile()

out = app.invoke({"messages": [HumanMessage("refund please")], "customer_id": "C-1",
                  "customer_tier": "unknown", "intent": "unknown",
                  "refund_amount_usd": None, "approval_status": None})
assert len(out["messages"]) >= 3      # human + 2 AI acks -> proves append, not overwrite
assert out["intent"] == "refund_request"
print("OK - messages appended to", len(out["messages"]), "; scalars overwrote cleanly")

---
## Exercise 3 - Conditional edge boundary test

**Task.** Implement the pure router route_after_propose(state) with AcmeCorp's rule: non-refund intents short-circuit to confirm_to_customer; refunds >= $500 route to request_human_approval; sub-threshold refunds go to auto_execute. Then unit-test it across refund_amount_usd in {0, 499.99, 500.00, 500.01, None}.

**Success criteria**
- 500.00 -> request_human_approval (the boundary lands on the human side)
- 499.99 -> auto_execute and None -> auto_execute (not a KeyError)
- The or 0.0 default is present and justified; a non-refund intent short-circuits to confirm_to_customer


In [ ]:
# Worked solution - the pure router + a boundary sweep. No LLM, no I/O: a router
# must stay pure because LangGraph may replay it during time-travel or after a resume.
HUMAN_APPROVAL_THRESHOLD_USD = 500.0

def route_after_propose(state) -> str:
    if state["intent"] != "refund_request":
        return "confirm_to_customer"                 # short-circuit non-refunds
    amount = state.get("refund_amount_usd") or 0.0    # `or 0.0` guards a missing/None field
    if amount >= HUMAN_APPROVAL_THRESHOLD_USD:
        return "request_human_approval"               # HITL branch
    return "auto_execute"                             # sub-threshold branch

cases = {0: "auto_execute", 499.99: "auto_execute", 500.00: "request_human_approval",
         500.01: "request_human_approval", None: "auto_execute"}
for amt, want in cases.items():
    got = route_after_propose({"intent": "refund_request", "refund_amount_usd": amt})
    assert got == want, f"{amt} -> {got}, expected {want}"
# non-refund intent short-circuits regardless of amount, and None never raises:
assert route_after_propose({"intent": "billing_question", "refund_amount_usd": 9999}) == "confirm_to_customer"
print("OK - boundary holds: 500.00 -> human approval; 499.99 / None -> auto_execute (no KeyError)")

---
## Exercise 4 - SQLite resume across a process restart

**Task.** Compile the escalation graph with a SqliteSaver pointed at a file on disk (check_same_thread=False). Invoke a thread partway, then simulate a crash by exiting the process. In a fresh process, re-instantiate the graph against the same DB file and re-invoke with the same thread_id - execution must continue from the last checkpoint, not restart.

**Success criteria**
- The resumed run reads customer_tier / intent set BEFORE the restart - proof the checkpointer persisted them
- The same thread_id continues from the last checkpoint, not from zero
- Swapping in InMemorySaver makes the second process restart - the "InMemory in production" catastrophe - and you can explain the contrast


In [ ]:
# Worked solution - the full six-node AcmeCorp escalation graph, compiled on a durable
# SqliteSaver, then resumed in a fresh graph object against the same DB file.
# EscalationState (Ex 2), route_after_propose + HUMAN_APPROVAL_THRESHOLD_USD (Ex 3) are reused.
import sqlite3, tempfile

# classify_intent is deterministic here (keyword lookup) so the notebook runs keyless; in
# production it is a Sonnet 5 call (see solutions/exercise_2_acmecorp_escalation_graph.py).
def classify_intent(state) -> dict:
    last = state["messages"][-1].content.lower()
    if "refund" in last:
        return {"intent": "refund_request"}
    if "invoice" in last or "bill" in last:
        return {"intent": "billing_question"}
    return {"intent": "feature_help"}

def retrieve_account(state) -> dict:                # stub CRM lookup (MCP call in M5.5)
    cid = state["customer_id"]
    tier = "enterprise" if cid.endswith("-ent") else "pro" if cid.endswith("-pro") else "free"
    return {"customer_tier": tier}

def propose_resolution(state) -> dict:
    if state["intent"] != "refund_request":
        return {"refund_amount_usd": None}
    import re as _re
    m = _re.search(r"\$?(\d+(?:\.\d+)?)", state["messages"][-1].content)
    return {"refund_amount_usd": float(m.group(1)) if m else 0.0}

def request_human_approval(state) -> dict:
    decision = interrupt({"customer_id": state["customer_id"],
                          "refund_amount_usd": state["refund_amount_usd"],
                          "policy": "requires_human_above_$500"})
    return {"approval_status": "approved" if str(decision).startswith("approve") else "rejected"}

def auto_execute(state) -> dict:
    return {"approval_status": "approved"}

def confirm_to_customer(state) -> dict:
    amount = state.get("refund_amount_usd") or 0.0
    if state["intent"] == "refund_request" and state.get("approval_status") == "approved":
        reply = f"Refund of ${amount:.2f} has been approved and issued."
    elif state["intent"] == "refund_request":
        reply = f"Your refund request for ${amount:.2f} was reviewed and not approved."
    else:
        reply = "Thanks - a support agent will follow up shortly."
    return {"messages": [AIMessage(reply)]}     # append the reply; EscalationState has no extra channel

def build_escalation_graph(checkpointer):
    g = StateGraph(EscalationState)
    for name, fn in [("classify_intent", classify_intent), ("retrieve_account", retrieve_account),
                     ("propose_resolution", propose_resolution),
                     ("request_human_approval", request_human_approval),
                     ("auto_execute", auto_execute), ("confirm_to_customer", confirm_to_customer)]:
        g.add_node(name, fn)
    g.add_edge(START, "classify_intent")
    g.add_edge("classify_intent", "retrieve_account")
    g.add_edge("retrieve_account", "propose_resolution")
    g.add_conditional_edges("propose_resolution", route_after_propose,
        {"request_human_approval": "request_human_approval", "auto_execute": "auto_execute",
         "confirm_to_customer": "confirm_to_customer"})
    g.add_edge("request_human_approval", "confirm_to_customer")
    g.add_edge("auto_execute", "confirm_to_customer")
    g.add_edge("confirm_to_customer", END)
    return g.compile(checkpointer=checkpointer)

def make_sqlite_checkpointer(path):
    try:
        from langgraph.checkpoint.sqlite import SqliteSaver
        conn = sqlite3.connect(path, check_same_thread=False)   # False is mandatory across threads
        return SqliteSaver(conn), "SqliteSaver"
    except Exception as exc:                                     # keyless / offline fallback
        print(f"[offline] SqliteSaver unavailable ({exc.__class__.__name__}); using InMemorySaver - "
              "true resume-across-restart only holds with the real SqliteSaver on disk")
        return InMemorySaver(), "InMemorySaver"

DB_PATH = os.path.join(tempfile.gettempdir(), "acme_escalation_practice.sqlite")
try:
    os.remove(DB_PATH)
except OSError:
    pass

cp1, kind = make_sqlite_checkpointer(DB_PATH)
app_p1 = build_escalation_graph(cp1)
cfg = {"configurable": {"thread_id": "acme-restart-01"}}
seed = {"messages": [HumanMessage("refund $620 for the renewal charged after I cancelled")],
        "customer_id": "cust-9-pro", "customer_tier": "unknown", "intent": "unknown",
        "refund_amount_usd": None, "approval_status": None}
first = app_p1.invoke(seed, config=cfg)             # runs up to the HITL interrupt, then pauses
print(f"[{kind}] process 1 paused at HITL?", "__interrupt__" in first,
      "| tier set before restart:", first.get("customer_tier"))

# --- simulate the process dying, then a FRESH process re-opening the SAME db file ---
if kind == "SqliteSaver":
    cp2, _ = make_sqlite_checkpointer(DB_PATH)      # brand-new saver over the same file
    app_p2 = build_escalation_graph(cp2)            # brand-new graph object
    resumed = app_p2.invoke(Command(resume="approved"), config=cfg)   # same thread_id -> continues
    print("process 2 resumed from checkpoint -> approval:", resumed.get("approval_status"))
    print("customer_tier survived the restart:", resumed.get("customer_tier"))
else:
    print("(InMemorySaver cannot survive a new process - install langgraph to watch the resume)")

---
## Exercise 5 - The full AcmeCorp escalation, paused and resumed

**Task.** Build the complete graph: classify_intent -> retrieve_account -> propose_resolution -> [conditional] -> request_human_approval | auto_execute -> confirm_to_customer. The approval node must call interrupt(payload); the caller resumes with Command(resume="approved"). Demonstrate BOTH a sub-threshold run that auto-executes end-to-end AND an above-threshold run that pauses at the interrupt and then resumes.

**Success criteria**
- Run A: intent=refund_request, amount=120.00, approval=approved, reply starts "Refund of $120.00..."
- Run B: the first invoke returns a dict containing __interrupt__; after Command(resume="approved"), approval_status=approved and the reply reads "Refund of $620.00 has been approved..."
- Compile without a checkpointer and interrupt() raises - HITL is a property of compile(checkpointer=...), not the node


In [ ]:
# Worked solution - reuse build_escalation_graph (Ex 4); it already wires all six nodes,
# the conditional edge, and the interrupt() at request_human_approval.
demo_app = build_escalation_graph(InMemorySaver())

def _seed(text, cid):
    return {"messages": [HumanMessage(text)], "customer_id": cid, "customer_tier": "unknown",
            "intent": "unknown", "refund_amount_usd": None, "approval_status": None}

# --- Run A: sub-threshold, auto-executes end to end ---
cfg_a = {"configurable": {"thread_id": "acme-low-A"}}
run_a = demo_app.invoke(_seed("Please refund $120 - duplicate charge on invoice 4421", "cust-1-pro"), cfg_a)
print("Run A:", run_a["intent"], f"${run_a['refund_amount_usd']:.2f}", run_a["approval_status"])
print("      reply:", run_a["messages"][-1].content)     # the reply lives on the messages channel
assert run_a["approval_status"] == "approved" and "__interrupt__" not in run_a

# --- Run B: above threshold, pauses at interrupt(), then resumes ---
cfg_b = {"configurable": {"thread_id": "acme-high-B"}}
run_b = demo_app.invoke(_seed("Refund $620 for the Pro renewal charged after I cancelled", "cust-2-pro"), cfg_b)
print("Run B interrupted?", "__interrupt__" in run_b)
run_b_final = demo_app.invoke(Command(resume="approved"), config=cfg_b)   # 47 hours later, human approves
print("Run B resumed:", run_b_final["approval_status"], "|", run_b_final["messages"][-1].content)
assert run_b_final["approval_status"] == "approved"

# --- HITL durability is a compile(checkpointer=...) property, NOT a node property ---
try:
    build_escalation_graph(None).invoke(_seed("refund $900", "cust-3-pro"),
                                        {"configurable": {"thread_id": "no-cp"}})
    print("no-checkpointer run finished without a durable pause - interrupt() had no store to "
          "persist to, so it could not survive a restart; always compile with a checkpointer for HITL")
except Exception as exc:
    print("no-checkpointer path ->", exc.__class__.__name__, "(interrupt needs compile(checkpointer=...))")

---
## Exercise 6 - Time-travel a bad classify_intent

**Task.** Run the graph end-to-end on a message the classifier gets wrong (it labels a refund as feature_help). Use get_state_history(cfg) to list checkpoints, pick the one right after classify_intent, patch intent to "refund_request" with app.update_state(...), then re-invoke with None to replay from there. Verify the new branch reaches request_human_approval.

**Success criteria**
- The replayed run routes to the HITL branch instead of the wrong self-serve reply
- The earlier retrieve_account work is NOT redone (you rewound past it, not before it)
- The router stays deterministic so the replay does not diverge from the original trace


In [ ]:
# Worked solution - run the graph, rewind to just after classify_intent, patch the
# intent, and replay forward. update_state + get_state_history are checkpointer features.
tt_app = build_escalation_graph(InMemorySaver())
cfg_tt = {"configurable": {"thread_id": "acme-tt-01"}}

# A message with NO refund/invoice/bill token -> keyword classify_intent mislabels it feature_help,
# so it never reaches the refund branch even though the customer clearly wants money back.
wrong = tt_app.invoke(
    {"messages": [HumanMessage("The renewal went through after I cancelled - I want my $700 back.")],
     "customer_id": "cust-7-pro", "customer_tier": "unknown", "intent": "unknown",
     "refund_amount_usd": None, "approval_status": None},
    config=cfg_tt)
print("first pass intent:", wrong["intent"], "| reply:", wrong["messages"][-1].content)

history = list(tt_app.get_state_history(cfg_tt))
print(len(history), "checkpoints recorded")

try:
    # find the checkpoint whose NEXT node is retrieve_account (i.e. right after classify_intent):
    target = None
    for snap in history:
        nxt = getattr(snap, "next", ())
        if nxt and "retrieve_account" in nxt:
            target = snap.config
            break
    if target is None:                       # fall back to a mid-history checkpoint
        target = history[len(history) // 2].config
    tt_app.update_state(target, {"intent": "refund_request"})   # patch state AT that checkpoint
    replayed = tt_app.invoke(None, config=cfg_tt)               # replay forward from there
    print("after time-travel -> intent:", replayed["intent"],
          "| approval:", replayed.get("approval_status"), "(routed through request_human_approval)")
except AttributeError:
    print("[offline] update_state / snapshot.next are real-langgraph features not in the fallback "
          "engine - install langgraph to see the replay route the $700 refund to request_human_approval")

---
## Exercise 7 - The Store: memory that outlives the thread

**Task.** The SqliteSaver checkpointer from Exercise 4 is thread-scoped SHORT-term memory - each thread_id gets its own state. AcmeCorp wants a returning customer's saved preference (say, "contact me by email, never phone") to survive into a brand-new support conversation next week. That is cross-thread LONG-term memory, and the checkpointer cannot do it - you need a Store. Compile the graph with BOTH: a checkpointer AND an InMemoryStore. Add a save_preference node that writes under a per-customer namespace, and a greet_returning node that reads it back. Prove a value written in thread acme-t1 is readable from a DIFFERENT thread acme-t2 for the same customer_id.

**Success criteria**
- Thread acme-t2 reads the channel="email" preference that thread acme-t1 wrote, even though the two threads share NO checkpoint
- Drop the store= argument from compile() and the injected store is None - long-term memory is a compile-time capability, like the checkpointer
- You can articulate checkpointer (short-term, thread-scoped) vs Store (long-term, cross-thread, namespace-keyed)


In [ ]:
# Worked solution - the Store is keyed by a per-CUSTOMER namespace, so a value written
# while serving one thread is readable while serving any other thread for that customer.
try:
    from langgraph.store.memory import InMemoryStore
    store_kind = "InMemoryStore (langgraph)"
except Exception:
    class InMemoryStore:                       # tiny offline shim with the same put/get surface
        def __init__(self):
            self._d = {}
        def put(self, ns, key, value):
            self._d[(tuple(ns), key)] = value
        def get(self, ns, key):
            v = self._d.get((tuple(ns), key))
            return type("Item", (), {"value": v})() if v is not None else None
    store_kind = "offline InMemoryStore shim"

store = InMemoryStore()
CUST = "cust-42-ent"

def save_preference(customer_id, store):
    ns = ("customer_prefs", customer_id)        # namespace is per-CUSTOMER, not per-thread
    store.put(ns, "contact_channel", {"channel": "email"})

def greet_returning(customer_id, store):
    ns = ("customer_prefs", customer_id)
    item = store.get(ns, "contact_channel")     # None on a first-ever customer
    if item is None:
        return "Welcome to AcmeCorp support."
    return f"Welcome back - contacting you by {item.value['channel']} as you asked."

# Thread acme-t1 writes the preference; thread acme-t2 (a different conversation, SAME
# customer) reads it back - something a thread-scoped checkpoint can never surface.
save_preference(CUST, store)                    # happens while serving thread acme-t1
print(f"[{store_kind}] thread acme-t2 sees:", greet_returning(CUST, store))
print("first-ever customer sees:", greet_returning("cust-brand-new", store))
assert greet_returning(CUST, store).startswith("Welcome back")

# Canonical wiring with real langgraph - store is a compile()-time capability, like the checkpointer:
#     app = graph.compile(checkpointer=SqliteSaver(conn), store=store)
#     def save_preference(state, *, store): store.put(("customer_prefs", state["customer_id"]), ...)
# LangGraph injects `store` into any node that declares a `store: BaseStore` parameter.

---
## Exercise 8 - Streaming: updates vs messages

**Task.** app.invoke(...) blocks until the whole graph finishes - useless for a support console that needs to show live progress AND stream the drafted reply as it types. Run the SAME above-threshold escalation twice with app.stream(...): once with stream_mode="updates" and once with stream_mode="messages", printing what each yields. Then write one sentence mapping each mode to a concrete AcmeCorp UI surface.

**Success criteria**
- "updates" yields one dict per node keyed by node name - drive the internal step tracker / audit log with it
- "messages" yields (message_chunk, metadata) tuples token by token from the LLM nodes - stream the draft reply into the agent's textbox
- One sentence maps each mode to a concrete AcmeCorp UI surface; "values" and list-mode multiplexing are mentioned


In [ ]:
# Worked solution - stream the SAME run two ways; each mode maps to a different UI surface.
stream_app = build_escalation_graph(InMemorySaver())
inp = {"messages": [HumanMessage("refund $80 for order 3311")], "customer_id": "cust-8-pro",
       "customer_tier": "unknown", "intent": "unknown", "refund_amount_usd": None,
       "approval_status": None}

print("--- stream_mode='updates' (one state-delta per node; drive the audit log) ---")
for chunk in stream_app.stream(inp, {"configurable": {"thread_id": "acme-str-upd"}}, stream_mode="updates"):
    for node_name, delta in chunk.items():
        print(f"  -> {node_name}: {list(delta.keys())}")

print("--- stream_mode='messages' (token-by-token from the LLM nodes; stream into the reply box) ---")
try:
    saw = False
    for token, meta in stream_app.stream(inp, {"configurable": {"thread_id": "acme-str-msg"}},
                                         stream_mode="messages"):
        saw = True
        print(getattr(token, "content", token), end="")
    print("\n(streamed)" if saw else "(no token stream)")
except (ValueError, TypeError):
    print("[offline] the fallback engine yields node deltas, not token tuples; with real langgraph "
          "'messages' streams (message_chunk, metadata) from the classify / propose / confirm nodes")

# Mapping each mode to an AcmeCorp UI surface:
#   'updates'                -> the internal step tracker / audit log (which node is firing now)
#   'messages'               -> the agent reply box (the draft reply streams in token by token)
#   'values'                 -> the FULL accumulated state after each step (a debug panel)
#   ['updates','messages']   -> multiplex both into one stream

---
## Grading rubric + submission checklist

| Criterion | Points | What full marks looks like |
|---|---|---|
| Minimum graph + reducer | 12 | Two invocations append; a plain `list` annotation collapses to 1 message, and you can explain why |
| EscalationState fields | 14 | Six fields, per-field reducer comment, a passing append test on `messages` |
| Conditional edge boundary | 18 | `500.00` routes to human approval; `None` does not raise; the `or 0.0` default is present and justified |
| SQLite resume | 18 | State survives a fresh process on the same `thread_id`; InMemory contrast demonstrated |
| Full escalation + HITL | 20 | Both runs work; `__interrupt__` appears then `Command(resume=...)` continues; no-checkpointer raises |
| Time-travel replay | 18 | Rewind to the classify checkpoint, patch `intent`, replay to the HITL node without redoing prior work |
| The Store (cross-thread memory) | 16 | Value written in thread `acme-t1` is read back from `acme-t2`; checkpointer (short-term) vs Store (long-term) articulated; `store=` is a `compile()` capability |
| Streaming modes | 14 | `"updates"` and `"messages"` both demonstrated and each mapped to a UI surface; `"values"` / list-mode multiplexing mentioned |

**Pass mark: 91/130 (70%).** A distinction (117+, 90%) also adds: a unit test proving the router NEVER calls an
LLM (replay-safe), a checkpointer-choice memo for AcmeCorp's launch (SQLite pilot -> AsyncPostgresSaver above
~10K threads, a directional rule of thumb), a note on when to reach for the Store's `PostgresStore`, and a
timeout-sweep sketch that re-invokes stale paused threads with `Command(resume="timed_out")`.

**Feedback SLA** - submit in the cohort Discord `#practice-labs` channel; rubric-graded feedback is returned
within **24 hours** (Plus-tier mentor review). Self-paced learners use this worked notebook and the rubric to
self-assess right away.
